# 01 — Integral boundary-layer chain: verification against exact solutions

The `blipb.ibl` chain (Thwaites laminar → Michel transition → Head turbulent,
axisymmetric via the Mangler transform, Squire–Young far-wake drag) is verified
here against the canonical flat-plate solutions, then run on the SPEC baseline
fuselage. The same checks run in CI with hard tolerances
(`tests/test_ibl_verification.py`).


In [ ]:
# resolve the repo root so the notebook runs from notebooks/ or the repo root
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "studies"))
import plotstyle

plotstyle.apply()
import matplotlib.pyplot as plt
import numpy as np

from blipb.ibl.head import solve_head
from blipb.ibl.thwaites import solve_thwaites

NU, U = 1.5e-5, 50.0  # air-like kinematic viscosity, edge velocity [m/s]


In [ ]:
# Laminar flat plate: Thwaites vs the exact Blasius skin friction
x = np.linspace(1e-4, 1.0, 800)
lam = solve_thwaites(x, np.full_like(x, U), NU)
re_x = U * x / NU
cf_blasius = 0.664 / np.sqrt(re_x)

band = re_x > 1e4
err = np.max(np.abs(lam.cf[band] / cf_blasius[band] - 1))
print(f"max |cf/cf_Blasius - 1| for Re_x > 1e4: {err:.2%}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.loglog(re_x, lam.cf, label="Thwaites (this work)")
ax.loglog(re_x, cf_blasius, "--", color="0.3", label="Blasius exact")
ax.set_xlabel(r"$Re_x$"); ax.set_ylabel(r"$C_f$"); ax.legend()
ax.set_title("laminar flat plate");


In [ ]:
# Turbulent flat plate: Head + Ludwieg-Tillmann vs power-law and Schultz-Grunow
xt = np.geomspace(0.09, 30.0, 700)
turb = solve_head(xt, np.full_like(xt, U), NU,
                  theta0=0.036 * 0.09 * (U * 0.09 / NU) ** -0.2, H0=1.40)
re_xt = U * xt / NU
cf_pow = 0.0592 * re_xt ** -0.2
cf_sg = 0.370 * np.log10(re_xt) ** -2.584

band = (re_xt > 1e6) & (re_xt < 1e8)
print(f"max |cf/cf_powerlaw - 1|, 1e6 < Re_x < 1e8: "
      f"{np.max(np.abs(turb.cf[band] / cf_pow[band] - 1)):.2%}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.loglog(re_xt, turb.cf, label="Head + L-T (this work)")
ax.loglog(re_xt, cf_pow, "--", color="0.3", label=r"$0.0592\,Re_x^{-1/5}$")
ax.loglog(re_xt, cf_sg, ":", label="Schultz-Grunow")
ax.set_xlabel(r"$Re_x$"); ax.set_ylabel(r"$C_f$"); ax.legend()
ax.set_title("turbulent flat plate");


In [ ]:
# The SPEC baseline fuselage (STARC-ABL class, M0.785 / FL350)
from blipb import BLIComparator

comp = BLIComparator()
bl = comp.bl
print(f"Re_L = {comp.flight.reynolds(comp.fuselage.length):.2e}")
print(f"transition at x/L = {bl.x_tr / comp.fuselage.length:.3f} (Michel)")
print(f"TE state: theta = {bl.theta_te*1e3:.1f} mm, H = {bl.h_te:.2f}")
print(f"Squire-Young far-wake profile drag = {bl.drag/1e3:.2f} kN")
print(f"separated: {bl.separated}")

fig, ax1 = plt.subplots(figsize=(6, 3))
ax1.plot(bl.x, bl.theta * 1e3)
ax1.set_xlabel("x [m]"); ax1.set_ylabel(r"$\theta$ [mm]", color="C0")
ax2 = ax1.twinx()
ax2.plot(bl.x, bl.H, color="C1")
ax2.set_ylabel("H", color="C1"); ax2.grid(False)
ax1.axvline(bl.x_tr, ls=":", color="0.4")
ax1.set_title("fuselage boundary layer (dotted line: transition)");


The full tolerance set (Blasius within 1.2%, Falkner–Skan stagnation flow,
1/7-power profile, axisymmetric Mangler consistency) is asserted in
`tests/test_ibl_verification.py`; Fig. 2 of the paper is the publication
version of these panels (`studies/figures.py::fig2_verification`).
